# 17 · DPO vs RLHF (and where PPO fits)

**Recap (13–16):** DPO trains a policy directly on **(chosen, rejected)** pairs, with
the reward written from the policy's own log-probs — no separate reward model. But the
*classic* way to learn from preferences is **RLHF**, which DPO was designed to
simplify. This notebook compares them, and places **PPO** (notebook 18) on the map.

In [ ]:
import numpy as np
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Step 1 — The piece DPO removes: the *reward model*

In classic RLHF you don't optimize against preferences directly. You first train a
**separate model — the reward model (RM)** — to imitate them: feed it a
(prompt, response) and it outputs a single **score**. You fit it on preference pairs
with the **Bradley-Terry loss** from notebook 15, so `score(chosen) > score(rejected)`.
Once trained, the RM can score **any** response — even brand-new ones.

Here's a one-parameter RM learning to rank, just to make it concrete. Each response is
summarized by a quality feature `x`; pairs have `x_chosen > x_rejected`:

In [ ]:
pairs = [(0.9, 0.2), (0.8, 0.5), (1.0, 0.3), (0.7, 0.4)]   # (x_chosen, x_rejected)

w = 0.0          # the reward model:  r(x) = w * x   (one parameter, for illustration)
lr = 1.0
for _ in range(500):
    g = 0.0
    for xc, xr in pairs:
        gap = w * xc - w * xr                  # score gap
        g += -(1 - sigmoid(gap)) * (xc - xr)   # gradient of  -log sigmoid(gap)  (Bradley-Terry)
    w -= lr * g / len(pairs)

print("learned reward weight w =", round(w, 2), " (positive -> higher quality scores higher)")
for xc, xr in pairs:
    print(f"  r(chosen)={w*xc:.2f}  >  r(rejected)={w*xr:.2f} ?  {w*xc > w*xr}")

The RM learned to score every chosen response above its rejected partner. **That trained
RM is what PPO then optimizes against.**

## Step 2 — The two recipes

**Classic RLHF (the InstructGPT recipe) — 3 stages:**
1. **SFT** — imitate demonstrations.
2. **Train the reward model** on preference pairs (Step 1).
3. **PPO** — the policy generates responses, the RM scores them, and reinforcement
   learning nudges the policy toward higher scores — with a **KL leash** to the SFT
   model so it can't drift into gibberish or game the RM.

**DPO — skip stages 2 and 3 as separate pieces.** The DPO algebra shows the policy
RLHF converges to satisfies `reward = β·log(π/π_ref) + const` (notebook 16). Substitute
that into the *same* Bradley-Terry loss and **the reward model cancels out** — leaving a
loss in the policy's own log-probs. You train directly on the pairs, like supervised
learning.

> **RLHF** = "train a judge, then practice against it by trial-and-error (RL)."
> **DPO** = "algebra proves you can skip building the judge and learn straight from the comparisons."

## Step 3 — Side by side

| | **RLHF** (PPO + reward model) | **DPO** |
|---|---|---|
| Separate reward model? | **yes** (an extra trained model) | **no** (reward is implicit) |
| Models in memory | policy + reference + reward (+ critic) ≈ **3–4** | policy + frozen reference = **2** |
| Training style | **online RL**: sample → score → update | **offline**: supervised loss on fixed pairs |
| Samples fresh responses while training? | **yes** (on-policy exploration) | no (bounded by the dataset) |
| Stability / tuning | finicky: clip, KL coef, value loss, **reward hacking** | stable, few knobs — like SFT |
| Compute | heavy (generation loop + many models) | light (batched gradient steps) |
| Can reward *novel* responses? | yes (RM generalizes) | only what the pairs cover |

## Step 4 — When each wins (honestly)

- **DPO** is the better *default*: simpler, cheaper, more stable, great with a fixed good
  preference set. Limits: **bounded by the dataset** (can't explore), sensitive to the
  reference, and has known failure modes (can push *both* chosen and rejected
  log-probs down).
- **RLHF/PPO** buys **online exploration** — the policy can discover high-reward
  responses beyond the dataset, and a reusable RM can score anything. The price is
  complexity, compute, instability, and reward-hacking risk.

DPO didn't make RLHF obsolete; it made the *common case* (align to a fixed preference
set) far cheaper.

## Step 5 — Where this lab sits

The lab replaces the **learned reward model** with a **programmatic execution reward**
(run the SQL: `1.0 / 0.3 / 0.0`, plus the cost tiebreak). For SQL that's a *perfect, free*
judge — no humans, no RM to train. So the methods line up on a spectrum:

| approach | reward source | online? | in the lab |
|---|---|---|---|
| **DPO** | execution reward → fixed pairs | offline | notebooks 13–16, `train_dpo` |
| **PPO + execution reward** | the SQL engine, live | online | `train_ppo` — **notebook 18** |
| **classic RLHF** | a *learned* reward model | online | the optional fork in `train_ppo` |

**The reward model is exactly the piece text-to-SQL makes unnecessary** — which is why
it's such a clean testbed. PPO here is "RLHF-style training, but the SQL engine *is* the
reward model."

## Recap

- A **reward model** scores any response; RLHF trains one (Bradley-Terry), then PPO
  optimizes the policy against it with a KL leash.
- **DPO** algebraically removes the RM and the RL loop — train the policy directly on pairs.
- DPO = simpler/cheaper/stabler default; RLHF = online exploration at higher cost.
- This lab swaps the learned RM for the **execution reward**.

**BAM!** Next: **`18 — PPO`**, the online method itself.